# Aurora Inference + Rollout (Fine-Tuned Model)

This notebook provides inference/rollout using a **fine-tuned Aurora checkpoint**.

Workflow:
1. Load YAML config
2. Load test dataset
3. Initialize model with fine-tuned checkpoint
4. Run rollout predictions
5. Save predictions and plots

## Environment / Imports

In [1]:
# Setup: Add project root to Python path
import os
import sys
from pathlib import Path

os.environ.setdefault('HF_HUB_DISABLE_PROGRESS_BARS', '1')

# Get the project root (parent of finetune directory)
notebook_dir = Path.cwd()
if notebook_dir.name == 'finetune':
    project_root = notebook_dir.parent
else:
    project_root = notebook_dir

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f'Project root: {project_root}')

Project root: /data/aurora


In [2]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import xarray as xr
from tqdm.auto import tqdm

import finetune.aurora_finetune_utils as ft
from aurora import (
    Aurora,
    Aurora12hPretrained,
    AuroraAirPollution,
    AuroraHighRes,
    AuroraPretrained,
    AuroraSmallPretrained,
    AuroraWave,
)

/home/azureuser/miniforge3/envs/aurora/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

In [3]:
# ============================================================================
# CONFIGURATION: Edit these settings
# ============================================================================

# Configuration file (auto-detects location)
CONFIG_PATH_NAME = 'aurora_O3_global_finetune_3day_lead_config.yaml'
if Path(CONFIG_PATH_NAME).exists():
    CONFIG_PATH = Path(CONFIG_PATH_NAME)
elif (Path('finetune') / CONFIG_PATH_NAME).exists():
    CONFIG_PATH = Path('finetune') / CONFIG_PATH_NAME
else:
    raise FileNotFoundError(f'Config file {CONFIG_PATH_NAME} not found')

# Path to your fine-tuned checkpoint and output dir.
# Defaults derive from the YAML's `case_name` so artifacts land under
# outputs/<case_name>/ and outputs/checkpoints/<case_name>/, matching
# what aurora_finetune_distributed.py wrote during training. Set these
# explicitly to override.
CHECKPOINT_PATH = None  # None = outputs/checkpoints/<case_name>/last.ckpt

# Rollout configuration (can override config YAML).
# For the O3 US-WEST config, 6 rollout steps at 12 hours/step is a 3-day forecast.
ROLLOUT_NUM_STEPS = None  # None = use config value, then max(data.target_lead_times) fallback

# Ensemble inference (Flow Matching is a generative model): number of
# stochastic rollout members to draw per initialization. Members differ
# only when the flow-refine model samples with sampling_steps > 1; with
# sampling_steps == 1 the model is deterministic and members are identical.
NUM_ENSEMBLE = 10   # set > 1 to generate an ensemble
ENSEMBLE_SEED = 0  # base RNG seed; ensemble member m uses ENSEMBLE_SEED + m
# Override the flow-refine sampling steps at inference time, independent of
# what training saved in the checkpoint. None = use the checkpoint's value.
# Set > 1 to force multi-step stochastic Flow-Matching sampling (required for
# ensemble spread) even if the checkpoint was saved as deterministic (1-step).
SAMPLING_STEPS_OVERRIDE = 8

# Output settings
SAVE_NETCDF = True
SAVE_PLOTS = True
OUTPUT_DIR = None  # None = outputs/<case_name>/

# ============================================================================

cfg = ft.load_config(CONFIG_PATH)
_case = str(cfg.get('case_name', '')).strip()
if CHECKPOINT_PATH is None:
    CHECKPOINT_PATH = str(Path(cfg['paths']['checkpoint_dir']) / 'last.ckpt')
if OUTPUT_DIR is None:
    OUTPUT_DIR = cfg['paths']['output_dir']
print(f"Loaded config from: {CONFIG_PATH.resolve()}")
print(f"Case name: {_case}")
print(f"Test data: {cfg['paths']['test_data_path']}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Output dir: {OUTPUT_DIR}")


Loaded config from: /data/aurora/finetune/aurora_O3_global_finetune_3day_lead_config.yaml
Case name: O3_global_3day_lead
Test data: /data/aurora/data/O3_global_3day_lead/test.nc
Checkpoint: /data/aurora/finetune/outputs/checkpoints/O3_global_3day_lead/last.ckpt
Output dir: /data/aurora/finetune/outputs/O3_global_3day_lead


## Load Test Dataset

In [4]:
test_ds = ft.open_dataset(cfg['paths']['test_data_path'], cfg)

# Merge static variables from external pickle (lsm, z, slt) into the test dataset.
static_path = cfg['paths'].get('static_data_path', '')
if static_path:
    test_ds = ft.merge_external_static_vars(test_ds, static_path, cfg)
    print(f'Merged static vars from: {static_path}')

resolved_specs = ft.resolve_variable_specs(test_ds, cfg)
lon_periodic = ft.validate_longitude_consistency([test_ds], cfg)
lon_dim = str(cfg.get('data', {}).get('lon_dim', 'longitude'))
model_longitude = test_ds[lon_dim].values

print('Test dataset sizes:', test_ds.sizes)
print('Predictors:', [f"{s.dataset_name}->{s.aurora_name} ({s.kind})" for s in resolved_specs.predictors])
print('Targets:   ', [f"{s.dataset_name}->{s.aurora_name} ({s.kind})" for s in resolved_specs.targets])
print('Static:    ', [f"{s.dataset_name}->{s.aurora_name} ({s.kind})" for s in resolved_specs.static])

Merged static vars from: /data/cams/aurora-0.4-air-pollution-static.pickle
Test dataset sizes: Frozen({'time': 184, 'latitude': 451, 'longitude': 900, 'level': 13})
Predictors: ['t2m->2t (surf)', 'u10->10u (surf)', 'v10->10v (surf)', 'msl->msl (surf)', 'pm1->pm1 (surf)', 'pm2p5->pm2p5 (surf)', 'pm10->pm10 (surf)', 'tcco->tcco (surf)', 'tc_no->tc_no (surf)', 'tcno2->tcno2 (surf)', 'gtco3->gtco3 (surf)', 'tcso2->tcso2 (surf)', 'z->z (atmos)', 'u->u (atmos)', 'v->v (atmos)', 't->t (atmos)', 'q->q (atmos)', 'co->co (atmos)', 'no->no (atmos)', 'no2->no2 (atmos)', 'go3->go3 (atmos)', 'so2->so2 (atmos)']
Targets:    ['go3->go3 (atmos)', 'gtco3->gtco3 (surf)']
Static:     ['lsm->lsm (static)', 'z_static->z (static)', 'slt->slt (static)', 'static_ammonia->static_ammonia (static)', 'static_ammonia_log->static_ammonia_log (static)', 'static_co->static_co (static)', 'static_co_log->static_co_log (static)', 'static_nox->static_nox (static)', 'static_nox_log->static_nox_log (static)', 'static_so2->s

## Build Test Samples

In [5]:
# For rollout inference we only need input_time_steps history frames to seed the model.
# Unlike training, there is no target lead-time tail constraint, so every anchor
# with enough input history is a valid forecast initialization time.
time_dim = cfg.get('data', {}).get('time_dim', 'time')
input_steps = int(cfg.get('data', {}).get('input_time_steps', 2))
n_time = int(test_ds.sizes[time_dim])

rollout_start_samples = [
    {
        'sample_index': sample_idx,
        'anchor_index': anchor_idx,
        'history_indices': list(range(anchor_idx - input_steps + 1, anchor_idx + 1)),
        'target_indices': {},
    }
    for sample_idx, anchor_idx in enumerate(range(input_steps - 1, n_time))
]
print(f'Number of rollout initialization times: {len(rollout_start_samples)}')

if not rollout_start_samples:
    raise ValueError('No valid start positions in test dataset (need at least input_time_steps timesteps).')

first_anchor_time = np.datetime64(
    test_ds[time_dim].values[rollout_start_samples[0]['anchor_index']], 's'
)
last_anchor_time = np.datetime64(
    test_ds[time_dim].values[rollout_start_samples[-1]['anchor_index']], 's'
)
print(f'First initialization time: {first_anchor_time}')
print(f'Last initialization time:  {last_anchor_time}')
print('Example first history indices:', rollout_start_samples[0]['history_indices'])


Number of rollout initialization times: 183
First initialization time: 2024-07-01T12:00:00
Last initialization time:  2024-09-30T12:00:00
Example first history indices: [0, 1]


## Initialize Model

In [6]:
MODEL_REGISTRY = {
    'aurora': Aurora,
    'aurora_pretrained': AuroraPretrained,
    'aurora_small_pretrained': AuroraSmallPretrained,
    'aurora_12h_pretrained': Aurora12hPretrained,
    'aurora_highres': AuroraHighRes,
    'aurora_air_pollution': AuroraAirPollution,
    'aurora_wave': AuroraWave,
}

model_cfg = cfg['model']
variant = str(model_cfg.get('model_variant', 'aurora_pretrained')).lower()

if variant not in MODEL_REGISTRY:
    raise ValueError(f'Unsupported model variant: {variant}. Available: {sorted(MODEL_REGISTRY)}')

model_var_cfg = ft.derive_model_variable_config(resolved_specs, cfg)
model_kwargs = dict(model_cfg.get('model_kwargs', {}))

if 'patch_size' in model_cfg and 'patch_size' not in model_kwargs:
    model_kwargs['patch_size'] = int(model_cfg['patch_size'])

mixed_precision_mode = str(model_cfg.get('mixed_precision', 'none')).lower()
if 'autocast' not in model_kwargs:
    model_kwargs['autocast'] = mixed_precision_mode in {'bf16', 'bfloat16', 'fp16'}

model = MODEL_REGISTRY[variant](
    surf_vars=model_var_cfg['surf_vars'],
    static_vars=model_var_cfg['static_vars'],
    atmos_vars=model_var_cfg['atmos_vars'],
    **model_kwargs,
)

# Wrap with refinement heads if enabled in config (must match training architecture).
model = ft.maybe_wrap_conv_refine(model, cfg, resolved_specs, lon=model_longitude)
model = ft.maybe_wrap_flow_refine(model, cfg, resolved_specs, lon=model_longitude)

print(f'Initialized {variant} model (conv_refine={bool(model_cfg.get("conv_refine_enabled", False))}, flow_refine={bool(model_cfg.get("flow_refine_enabled", False))})')

Initialized aurora_air_pollution model (conv_refine=False, flow_refine=True)


## Load Fine-Tuned Checkpoint

In [7]:
import subprocess as _sp

checkpoint_path = Path(CHECKPOINT_PATH).expanduser()
if not checkpoint_path.exists():
    raise FileNotFoundError(f'Checkpoint not found: {checkpoint_path}')

# Pick the idle GPU with the most free memory using nvidia-smi
def _pick_best_gpu():
    try:
        out = _sp.check_output(
            ['nvidia-smi', '--query-gpu=index,memory.free',
             '--format=csv,noheader,nounits'], text=True)
    except Exception:
        return 0
    best_idx, best_free = 0, 0
    for line in out.strip().splitlines():
        parts = line.split(',')
        idx, free = int(parts[0]), float(parts[1])
        if free > best_free:
            best_idx, best_free = idx, free
    print(f'Auto-selected GPU {best_idx} ({best_free/1024:.1f} GB free)')
    return best_idx

if torch.cuda.is_available():
    device = torch.device(f'cuda:{_pick_best_gpu()}')
else:
    print('CUDA unavailable; falling back to CPU.')
    device = torch.device('cpu')

# Load checkpoint
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
ft.validate_checkpoint_longitude(model, checkpoint)

# Restore norm stats for flow-refine wrapper (if applicable). Prefer
# stats persisted in the checkpoint; fall back to recomputing from
# train data if the checkpoint predates norm-stat persistence or has
# stale level counts (e.g. old checkpoints computed stats over loss_levels
# only rather than the full atmos grid).
try:
    from finetune.flow_refine import AuroraFlowRefine as _AFR
    if isinstance(model, _AFR):
        ns = checkpoint.get('norm_stats')

        # Validate: atmos norm-stat level count must match loss_levels
        # (or full atmos_levels if loss_levels not specified).
        _full_levels = cfg.get("data", {}).get("atmos_levels", [])
        _target_vars = cfg.get("data", {}).get("target_variables", [])
        _loss_levels_map = {}
        for _tv in _target_vars:
            if isinstance(_tv, dict) and _tv.get("kind") == "atmos" and _tv.get("loss_levels"):
                _loss_levels_map[_tv.get("aurora_name", _tv.get("dataset_name"))] = _tv["loss_levels"]
        _stale = False
        if ns is not None and _full_levels:
            for _v, _vs in ns.items():
                _m = _vs.get("mean")
                if _m is not None and hasattr(_m, "numel") and _m.numel() > 1:
                    _expected = len(_loss_levels_map.get(_v, _full_levels))
                    if _m.numel() != _expected:
                        print(
                            f"norm_stats for {_v!r} has {_m.numel()} levels but "
                            f"expected {_expected}; recomputing..."
                        )
                        _stale = True
                        break

        if ns is None or _stale:
            if ns is None:
                print(f"Checkpoint has no norm_stats; recomputing from {cfg['paths']['train_data_path']}...")
            _train_ds = ft.open_dataset(cfg['paths']['train_data_path'], cfg)
            _static_path = cfg['paths'].get('static_data_path')
            if _static_path:
                _train_ds = ft.merge_external_static_vars(_train_ds, _static_path, cfg)
            _specs = ft.resolve_variable_specs(_train_ds, cfg)
            ns = ft.compute_target_normalization_stats(_train_ds, _specs, cfg)

        model.set_norm_stats(ns)
        print(f'Set flow-refine norm stats: {list(ns.keys())}')

        # Restore sampling_steps from checkpoint (training may have switched
        # to multi-step after the phase-fraction threshold; always matches
        # training phase when the checkpoint was saved).
        _saved_steps = checkpoint.get('flow_sampling_steps')
        if _saved_steps is not None and model.sampling_steps != _saved_steps:
            print(
                f'Restored sampling_steps: {model.sampling_steps} -> {_saved_steps}'
                f' (from checkpoint epoch {checkpoint.get("epoch", "?")})'
            )
            model.sampling_steps = int(_saved_steps)
except Exception as _e:
    print(f'flow_refine init skipped: {_e}')

# Inference-time override of the flow-refine sampling steps. Applied after the
# checkpoint's value is restored so it always takes precedence when set.
if SAMPLING_STEPS_OVERRIDE is not None:
    try:
        from finetune.flow_refine import AuroraFlowRefine as _AFR
        if isinstance(model, _AFR):
            _old_steps = model.sampling_steps
            model.sampling_steps = int(SAMPLING_STEPS_OVERRIDE)
            print(
                f'Overrode sampling_steps: {_old_steps} -> {model.sampling_steps} '
                '(inference-time SAMPLING_STEPS_OVERRIDE)'
            )
        else:
            print('SAMPLING_STEPS_OVERRIDE ignored: model is not flow-refine.')
    except Exception as _e:
        print(f'SAMPLING_STEPS_OVERRIDE skipped: {_e}')

model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

print(f'Loaded checkpoint from: {checkpoint_path}')
print(f'  Epoch: {checkpoint.get("epoch", "N/A")}')
print(f'  Best val loss: {checkpoint.get("best_val_loss", "N/A")}')
print(f'Device: {device}')


Auto-selected GPU 0 (93.1 GB free)
Set flow-refine norm stats: ['go3', 'gtco3']
Overrode sampling_steps: 1 -> 8 (inference-time SAMPLING_STEPS_OVERRIDE)
Loaded checkpoint from: /data/aurora/finetune/outputs/checkpoints/O3_global_3day_lead/last.ckpt
  Epoch: 49
  Best val loss: 0.0014061637921258807
Device: cuda:0


## Run Rollout Inference

In [8]:
from finetune.longitude import add_cyclic_column, dateline_discontinuity_from_edges

output_dir = Path(OUTPUT_DIR).expanduser()
output_dir.mkdir(parents=True, exist_ok=True)

def _timestamp_tag(value):
    return str(np.datetime64(value, 's')).replace('-', '').replace(':', '')

# Override rollout steps if specified.
if ROLLOUT_NUM_STEPS is not None:
    cfg['rollout']['rollout_num_steps'] = ROLLOUT_NUM_STEPS

rollout_steps = int(cfg['rollout'].get('rollout_num_steps', 0))
if rollout_steps <= 0:
    # Fall back to max(target_lead_times) from data config. For the O3/NO2 US-WEST
    # 3-day configs this is 6 steps, and rollout_step_hours is 12.
    lead_times = cfg.get('data', {}).get('target_lead_times', [])
    if lead_times:
        rollout_steps = max(int(x) for x in lead_times)
        cfg['rollout']['rollout_num_steps'] = rollout_steps
    else:
        rollout_steps = 7
        cfg['rollout']['rollout_num_steps'] = rollout_steps

rollout_step_hours = int(cfg.get('rollout', {}).get('rollout_step_hours', 12))
forecast_hours = rollout_steps * rollout_step_hours
smooth_sigma = float(cfg.get('rollout', {}).get('smooth_sigma', 0.0))
patch_size = int(cfg.get('model', {}).get('patch_size', 3))

n_ensemble = max(1, int(NUM_ENSEMBLE))

# Flow Matching only produces ensemble spread when sampling stochastically
# (sampling_steps > 1). Warn if an ensemble was requested on a deterministic model.
_flow_steps = int(getattr(model, 'sampling_steps', 1) or 1)
if n_ensemble > 1 and _flow_steps <= 1:
    print(
        f'WARNING: NUM_ENSEMBLE={n_ensemble} but model sampling_steps={_flow_steps} '
        '(deterministic) -> ensemble members will be identical. '
        'Use a flow-refine checkpoint with sampling_steps > 1 for ensemble spread.'
    )

rollout_results = []
with tqdm(
    rollout_start_samples,
    desc='Inference rollouts',
    unit='init',
    dynamic_ncols=True,
    leave=False,
    position=0,
    mininterval=1.0,
) as pbar:
    for rollout_idx, start_sample in enumerate(pbar):
        anchor_time = np.datetime64(test_ds[time_dim].values[start_sample['anchor_index']], 's')
        history_times = [
            np.datetime64(test_ds[time_dim].values[idx], 's')
            for idx in start_sample['history_indices']
        ]
        init_tag = _timestamp_tag(anchor_time)
        rollout_path = output_dir / f'rollout_predictions_init_{init_tag}.nc'

        # Draw n_ensemble stochastic rollouts for this initialization. Each member
        # uses a distinct seed so Flow Matching produces independent samples.
        member_datasets = []
        member_pred_counts = []
        for member_idx in range(n_ensemble):
            seed = int(ENSEMBLE_SEED) + member_idx
            torch.manual_seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed)

            pbar.set_postfix(
                init=str(anchor_time),
                member=f'{member_idx + 1}/{n_ensemble}',
                status='running',
                refresh=True,
            )

            predictions = ft.run_rollout(
                model=model,
                ds=test_ds,
                start_sample=start_sample,
                config=cfg,
                resolved_specs=resolved_specs,
                device=device,
            )
            member_pred_counts.append(len(predictions))

            if predictions and SAVE_NETCDF:
                member_ds = ft.save_predictions(
                    predictions,
                    rollout_path,
                    save_netcdf=False,
                    resolved_specs=resolved_specs,
                    smooth_sigma=smooth_sigma,
                    patch_size=patch_size,
                    lon_periodic=lon_periodic,
                )
                member_datasets.append(member_ds)

            del predictions
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        num_predictions = member_pred_counts[0] if member_pred_counts else 0
        result = {
            'rollout_idx': rollout_idx,
            'start_sample': start_sample,
            'anchor_time': anchor_time,
            'history_times': history_times,
            'num_predictions': num_predictions,
            'num_members': n_ensemble,
        }

        if member_datasets and SAVE_NETCDF:
            # Single member -> keep the flat (time, lat, lon) layout. Multiple
            # members -> stack along a new leading 'member' dimension.
            if len(member_datasets) == 1:
                rollout_ds = member_datasets[0]
            else:
                rollout_ds = xr.concat(member_datasets, dim='member', join='exact')
                rollout_ds = rollout_ds.assign_coords(member=np.arange(len(member_datasets)))

            rollout_ds.attrs['initialization_time'] = str(anchor_time)
            rollout_ds.attrs['anchor_index'] = int(start_sample['anchor_index'])
            rollout_ds.attrs['history_times'] = ','.join(str(t) for t in history_times)
            rollout_ds.attrs['num_ensemble_members'] = len(member_datasets)
            if lon_periodic:
                for _name, _da in rollout_ds.data_vars.items():
                    _edge = [_da.isel(longitude=_i).values for _i in (0, 1, -2, -1)]
                    _seam = dateline_discontinuity_from_edges(*_edge)
                    print(
                        f'{_name} dateline diagnostic: jump={_seam["seam_jump"]:.4g}, '
                        f'local_ratio={_seam["local_ratio"]:.3f}'
                    )
            for _coord in ('time', 'latitude', 'longitude', 'level'):
                if _coord in rollout_ds.coords:
                    rollout_ds[_coord].encoding['_FillValue'] = None
            rollout_ds.to_netcdf(str(rollout_path))
            rollout_ds.close()
            for member_ds in member_datasets:
                member_ds.close()

            result['rollout_path'] = rollout_path
            pbar.set_postfix(
                init=str(anchor_time),
                steps=num_predictions,
                members=len(member_datasets),
                status='saved',
                refresh=True,
            )
        else:
            pbar.set_postfix(
                init=str(anchor_time),
                steps=num_predictions,
                members=n_ensemble,
                status='done',
                refresh=True,
            )

        rollout_results.append(result)

print(
    f'Completed {len(rollout_results)} initialization(s); '
    f'{rollout_steps} steps ({forecast_hours} forecast hours) each, '
    f'{n_ensemble} ensemble member(s) per initialization.'
)
if SAVE_NETCDF and smooth_sigma > 0:
    print(f'Gaussian smoothing: sigma={smooth_sigma}')


Inference rollouts:   0%|          | 0/183 [00:00<?, ?init/s, init=2024-07-01T12:00:00, member=1/10, status=running]/home/azureuser/miniforge3/envs/aurora/lib/python3.14/contextlib.py:109: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
Inference rollouts:   0%|          | 0/183 [00:17<?, ?init/s, init=2024-07-01T12:00:00, member=2/10, status=running]/home/azureuser/miniforge3/envs/aurora/lib/python3.14/contextlib.py:109: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
Inference rollouts:   0%|          | 0/183 [00:34<?, ?init/s, init=2024-07-01T12:00:00, member=3/10, status=running]/home/

Completed 183 initialization(s); 6 steps (72 forecast hours) each, 10 ensemble member(s) per initialization.
Gaussian smoothing: sigma=1.0


## Save Predictions to NetCDF

In [9]:
# NetCDF files are saved inside the rollout loop above so each initialization
# can be written immediately without retaining all predictions in memory.


## Plot Results

In [10]:
saved_rollout_results = [result for result in rollout_results if 'rollout_path' in result]
if saved_rollout_results and SAVE_PLOTS:
    plot_vars = cfg.get('notebook', {}).get('plot_variables', [])

    for result in saved_rollout_results:
        rollout_ds = xr.open_dataset(result['rollout_path'])
        try:
            if not plot_vars:
                # Default: first 3 dataset variable names from the current rollout.
                plot_vars_for_result = list(rollout_ds.data_vars.keys())[:3]
            else:
                plot_vars_for_result = plot_vars

            init_tag = _timestamp_tag(result['anchor_time'])
            for var in plot_vars_for_result:
                if var not in rollout_ds:
                    print(f'Variable {var} not in rollout dataset for {init_tag}, skipping')
                    continue

                da = rollout_ds[var]
                # Collapse the ensemble dimension to its mean for plotting.
                if 'member' in da.dims:
                    da = da.mean('member')
                has_level = 'level' in da.dims

                # Plot each time step, capped for readability.
                n_times = da.sizes.get('time', 1)
                num_steps = min(n_times, 4)
                fig, axes = plt.subplots(1, num_steps, figsize=(5 * num_steps, 5))
                if num_steps == 1:
                    axes = [axes]

                for step_i, ax in enumerate(axes):
                    if has_level:
                        image = da.isel(time=step_i, level=-1)
                        level_val = float(da['level'].values[-1])
                        title = f'{var} | init={result["anchor_time"]} | t={step_i} | {level_val:.0f} hPa'
                    else:
                        image = da.isel(time=step_i)
                        title = f'{var} | init={result["anchor_time"]} | t={step_i}'

                    plot_lon, plot_values = add_cyclic_column(
                        da['longitude'].values, image.values, only_if_periodic=True,
                    )
                    mesh = ax.pcolormesh(
                        plot_lon, da['latitude'].values,
                        plot_values, cmap='viridis', shading='auto',
                    )
                    fig.colorbar(mesh, ax=ax, shrink=0.8)
                    ax.set_title(title)
                    ax.set_xlabel('longitude')
                    ax.set_ylabel('latitude')

                fig.tight_layout()
                fig_path = output_dir / f'rollout_init_{init_tag}_{var}.png'
                fig.savefig(fig_path, dpi=150, bbox_inches='tight')
                print(f'Saved plot: {fig_path}')
                plt.close(fig)
        finally:
            rollout_ds.close()
else:
    print('Skipping plots (SAVE_PLOTS=False or no saved rollout data)')


Saved plot: /data/aurora/finetune/outputs/O3_global_3day_lead/rollout_init_20240701T120000_go3.png
Saved plot: /data/aurora/finetune/outputs/O3_global_3day_lead/rollout_init_20240701T120000_gtco3.png
Saved plot: /data/aurora/finetune/outputs/O3_global_3day_lead/rollout_init_20240702T000000_go3.png
Saved plot: /data/aurora/finetune/outputs/O3_global_3day_lead/rollout_init_20240702T000000_gtco3.png
Saved plot: /data/aurora/finetune/outputs/O3_global_3day_lead/rollout_init_20240702T120000_go3.png
Saved plot: /data/aurora/finetune/outputs/O3_global_3day_lead/rollout_init_20240702T120000_gtco3.png
Saved plot: /data/aurora/finetune/outputs/O3_global_3day_lead/rollout_init_20240703T000000_go3.png
Saved plot: /data/aurora/finetune/outputs/O3_global_3day_lead/rollout_init_20240703T000000_gtco3.png
Saved plot: /data/aurora/finetune/outputs/O3_global_3day_lead/rollout_init_20240703T120000_go3.png
Saved plot: /data/aurora/finetune/outputs/O3_global_3day_lead/rollout_init_20240703T120000_gtco3.png


## Summary

In [11]:
print('\n' + '='*70)
print('INFERENCE SUMMARY')
print('='*70)
print(f'Checkpoint: {checkpoint_path}')
print(f'Initialization count: {len(rollout_results)}')
print(f'Rollout steps per initialization: {rollout_steps}')
print(f'Ensemble members per initialization: {NUM_ENSEMBLE}')
print(f'Forecast hours per initialization: {forecast_hours}')
print(f'Output directory: {output_dir}')
saved_paths = [str(result['rollout_path']) for result in rollout_results if 'rollout_path' in result]
print(f'NetCDF files saved: {len(saved_paths)}')
for path in saved_paths[:5]:
    print(f'  {path}')
if len(saved_paths) > 5:
    print(f'  ... {len(saved_paths) - 5} more')
print('='*70)



INFERENCE SUMMARY
Checkpoint: /data/aurora/finetune/outputs/checkpoints/O3_global_3day_lead/last.ckpt
Initialization count: 183
Rollout steps per initialization: 6
Ensemble members per initialization: 10
Forecast hours per initialization: 72
Output directory: /data/aurora/finetune/outputs/O3_global_3day_lead
NetCDF files saved: 183
  /data/aurora/finetune/outputs/O3_global_3day_lead/rollout_predictions_init_20240701T120000.nc
  /data/aurora/finetune/outputs/O3_global_3day_lead/rollout_predictions_init_20240702T000000.nc
  /data/aurora/finetune/outputs/O3_global_3day_lead/rollout_predictions_init_20240702T120000.nc
  /data/aurora/finetune/outputs/O3_global_3day_lead/rollout_predictions_init_20240703T000000.nc
  /data/aurora/finetune/outputs/O3_global_3day_lead/rollout_predictions_init_20240703T120000.nc
  ... 178 more
